<a href="https://colab.research.google.com/github/pop123-ux/HuggingFace-Project-Learning/blob/main/course/chapter6/Training_a_New_Tokenizer_from_an_Old_One.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Training a new tokenizer from an old one

Install the Transformers, Datasets, and Evaluate libraries to run this notebook.

In [1]:
!pip install datasets evaluate transformers[sentencepiece]
!apt install git-lfs

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 2.5 MB/s eta 0:00:00
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  git-lfs
0 upgraded, 1 newly installed, 0 to remove and 3 not upgraded.
Need to get 3,544 kB of archives.
After this operation, 10.5 MB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy-updates/universe amd64 git-lfs amd64 3.0.2-1ubuntu0.3 [3,544 kB]
Fetched 3,544 kB in 0s (10.4 MB/s)
Selecting previously unselected package git-lfs.
(Reading database ... 118243 files and directories currently installed.)
Preparing to unpack .../git-lfs_3.0.2-1ubuntu0.3_amd64.deb ...
Unpacking git-lfs (3.0.2-1ubuntu0.3) ...
Setting up git-lfs (3.0.2-1ubuntu0.3) ...
Processing triggers for man-db (2.10.2-1) ...


You will need to setup git, adapt your email and name in the following cell.

In [2]:
!git config --global user.email "alexandrupp55@gmail.com"
!git config --global user.name "Pop123-ux"

In [3]:
from huggingface_hub import notebook_login

notebook_login()

In [5]:
from datasets import load_dataset

raw_datasets = load_dataset("claudios/code_search_net", "python")

README.md:   0%|          | 0.00/13.6k [00:00<?, ?B/s]

python/train-00000-of-00003.parquet:   0%|          | 0.00/130M [00:00<?, ?B/s]

python/train-00001-of-00003.parquet:   0%|          | 0.00/135M [00:00<?, ?B/s]

python/train-00002-of-00003.parquet:   0%|          | 0.00/125M [00:00<?, ?B/s]

python/test-00000-of-00001.parquet:   0%|          | 0.00/21.7M [00:00<?, ?B/s]

python/validation-00000-of-00001.parquet:   0%|          | 0.00/23.1M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/412178 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/22176 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/23107 [00:00<?, ? examples/s]

In [6]:
raw_datasets

DatasetDict({
    train: Dataset({
        features: ['repository_name', 'func_path_in_repository', 'func_name', 'whole_func_string', 'language', 'func_code_string', 'func_documentation_string', 'func_code_url'],
        num_rows: 412178
    })
    test: Dataset({
        features: ['repository_name', 'func_path_in_repository', 'func_name', 'whole_func_string', 'language', 'func_code_string', 'func_documentation_string', 'func_code_url'],
        num_rows: 22176
    })
    validation: Dataset({
        features: ['repository_name', 'func_path_in_repository', 'func_name', 'whole_func_string', 'language', 'func_code_string', 'func_documentation_string', 'func_code_url'],
        num_rows: 23107
    })
})

In [11]:
raw_datasets['train'][123456]['whole_func_string']

'def updater():\n    """Update the current installation.\n\n    git clones the latest version and merges it with the current directory.\n    """\n    print(\'%s Checking for updates\' % run)\n    # Changes must be separated by ;\n    changes = \'\'\'major bug fixes;removed ninja mode;dropped python < 3.2 support;fixed unicode output;proxy support;more intels\'\'\'\n    latest_commit = requester(\'https://raw.githubusercontent.com/s0md3v/Photon/master/core/updater.py\', host=\'raw.githubusercontent.com\')\n    # Just a hack to see if a new version is available\n    if changes not in latest_commit:\n        changelog = re.search(r"changes = \'\'\'(.*?)\'\'\'", latest_commit)\n        # Splitting the changes to form a list\n        changelog = changelog.group(1).split(\';\')\n        print(\'%s A new version of Photon is available.\' % good)\n        print(\'%s Changes:\' % info)\n        for change in changelog: # print changes\n            print(\'%s>%s %s\' % (green, end, change))\n\n 

The first thing that we need to do is transform the dataset into an iterator of lists of texts -- for instance, a list of list of texts.

Using lists of texts will enable our tokenizer to go faster (training on batches of texts instead of processing individual texts one by one), and it should be an iterator if we want to avoid having everything in memory at once.

Using a python generator, we can avoid Python loading anything into memory until it's actually necessary.

To create such a generator, we need just to replace the brackets with parantheses:

In [ ]:
training_corpus = (
    raw_datasets['train'][i: i + 1000]['whole_func_string']
    for i in range(0, len(raw_datasets['train']), 1000)
)

This line of code does not fetch any elements of the dataset; it just creates an object we can use in a Python for loop.

The texts will only be loaded when we need them (that is, when we're at the step of the for loop that requires them), and only 1000 texts at a time will be loaded.

This way we won't exhaust all of our memory even if we are processing a huge dataset

The problem is that a generator object can only be used once. So, instead of giving us the list of the first digits twice:

In [12]:
gen = (i for i in range(10))
print(list(gen))
print(list(gen))

[0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
[]


We get them once and then an empty list..

That's why we define a function that returns a generator instead:

In [14]:
def get_training_corpus():
  return (
      raw_datasets['train'][i : i + 1000]['whole_func_string']
      for i in range(0, len(raw_datasets['train']), 1000)
  )

training_corpus = get_training_corpus()
training_corpus

<generator object get_training_corpus.<locals>.<genexpr> at 0x7a909017dee0>

We can also define our generator inside a for loop by using the yield statement:

In [17]:
def get_training_corpus():
  dataset = raw_datasets['train']
  for start_idx in range(0, len(dataset), 1000):
    samples = dataset[start_idx : start_idx + 1000]
    yield samples['whole_func_string']

  training_corpus = get_training_corpus()
  training_corpus

## Training a new tokenizer ##

Now that we have our corpus in the form of an iterator of batches of texts, we are ready to train a new tokenizer. To do this, we first need to load the tokenizer we want to pair with our model (here, GPT-2):

In [18]:
from transformers import AutoTokenizer

old_tokenizer = AutoTokenizer.from_pretrained("gpt2")

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

Let's first look at how this tokenizer would treat an example function:

In [19]:
example = '''def add_numbers(a, b):
    """Add the two numbers `a` and `b`."""
    return a + b'''

tokens = old_tokenizer.tokenize(example)
tokens

['def',
 'Ġadd',
 '_',
 'n',
 'umbers',
 '(',
 'a',
 ',',
 'Ġb',
 '):',
 'Ċ',
 'Ġ',
 'Ġ',
 'Ġ',
 'Ġ"""',
 'Add',
 'Ġthe',
 'Ġtwo',
 'Ġnumbers',
 'Ġ`',
 'a',
 '`',
 'Ġand',
 'Ġ`',
 'b',
 '`',
 '."',
 '""',
 'Ċ',
 'Ġ',
 'Ġ',
 'Ġ',
 'Ġreturn',
 'Ġa',
 'Ġ+',
 'Ġb']

This tokenizer has a few special symbols, like Ġ and Ċ, which can denote spaces and newlines, respectively.

As we can see, this is not too efficient: the tokenizer returns individual tokens for each space, when it could group together indentation levels (since having sets of four or eight spaces is going to be very common in code). It also split the function name a bit weirdly, not being used to seeing words with the _ character.

Let's train a new tokenizer and see if it solves these issues. For this, we'll use the method train_new_from_iterator():

In [20]:
tokenizer = old_tokenizer.train_new_from_iterator(training_corpus, 52000)

In [21]:
tokens = tokenizer.tokenize(example)
tokens

['def',
 'Ġadd',
 '_',
 'numbers',
 '(',
 'a',
 ',',
 'Ġb',
 '):',
 'ĊĠĠĠ',
 'Ġ"""',
 'Add',
 'Ġthe',
 'Ġtwo',
 'Ġnumbers',
 'Ġ`',
 'a',
 '`',
 'Ġand',
 'Ġ`',
 'b',
 '`."""',
 'ĊĠĠĠ',
 'Ġreturn',
 'Ġa',
 'Ġ+',
 'Ġb']

Here we again see the special symbols Ġ and Ċ that denote spaces and newlines, but we can also see that our tokenizer learned some tokens that are highly specific to a corpus of Python functions: for example, there is a ĊĠĠĠ token that represents an indentation, and a Ġ""" token that represents the three quotes that start a docstring.

The tokenizer also correctly split the function name on _. This is quite a compact representation; comparatively, using the plain English tokenizer on the same example will give us a longer sentence:

In [25]:
len(tokens), len(old_tokenizer.tokenize(example))

(27, 36)

Now, another example:

In [26]:
example = """class LinearLayer():
    def __init__(self, input_size, output_size):
        self.weight = torch.randn(input_size, output_size)
        self.bias = torch.zeros(output_size)

    def __call__(self, x):
        return x @ self.weights + self.bias
    """
tokenizer.tokenize(example)

['class',
 'ĠLinear',
 'Layer',
 '():',
 'ĊĠĠĠ',
 'Ġdef',
 'Ġ__',
 'init',
 '__(',
 'self',
 ',',
 'Ġinput',
 '_',
 'size',
 ',',
 'Ġoutput',
 '_',
 'size',
 '):',
 'ĊĠĠĠĠĠĠĠ',
 'Ġself',
 '.',
 'weight',
 'Ġ=',
 'Ġtorch',
 '.',
 'randn',
 '(',
 'input',
 '_',
 'size',
 ',',
 'Ġoutput',
 '_',
 'size',
 ')',
 'ĊĠĠĠĠĠĠĠ',
 'Ġself',
 '.',
 'bias',
 'Ġ=',
 'Ġtorch',
 '.',
 'zeros',
 '(',
 'output',
 '_',
 'size',
 ')',
 'ĊĊĠĠĠ',
 'Ġdef',
 'Ġ__',
 'call',
 '__(',
 'self',
 ',',
 'Ġx',
 '):',
 'ĊĠĠĠĠĠĠĠ',
 'Ġreturn',
 'Ġx',
 'Ġ@',
 'Ġself',
 '.',
 'weights',
 'Ġ+',
 'Ġself',
 '.',
 'bias',
 'ĊĠĠĠĠ']

In addition to the token corresponding to an indentation, here we can also see a token for a double indentation: ĊĠĠĠĠĠĠĠ.

The special Python words like class, init, call, self, and return are each tokenized as one token, and we can see that as well as splitting on _ and . the tokenizer correctly splits even camel-cased names: LinearLayer is tokenized as ["ĠLinear", "Layer"].

## Saving the tokenizer ##

To make sure we can use it later, we need to save our new tokenizer. Like for models, this can be done with the save_pretrained() method:

In [23]:
tokenizer.save_pretrained("code-search-net-tokenizer")

('code-search-net-tokenizer/tokenizer_config.json',
 'code-search-net-tokenizer/tokenizer.json')

In [27]:
tokenizer.push_to_hub('code-search-net-tokenizer')

CommitInfo(commit_url='https://huggingface.co/pop123ux/code-search-net-tokenizer/commit/0ff873e09909e348539a3f4645ff599c18cc23cd', commit_message='Upload tokenizer', commit_description='', oid='0ff873e09909e348539a3f4645ff599c18cc23cd', pr_url=None, repo_url=RepoUrl('https://huggingface.co/pop123ux/code-search-net-tokenizer', endpoint='https://huggingface.co', repo_type='model', repo_id='pop123ux/code-search-net-tokenizer'), pr_revision=None, pr_num=None)

This will create a new repository in our namespace with the name "code-search-net-tokenizer", containing the tokenizer file. We can load the tokenizer from anywhere with the from_pretrained() method:

In [28]:
tokenizer = AutoTokenizer.from_pretrained("pop123ux/code-search-net-tokenizer")

tokenizer_config.json:   0%|          | 0.00/315 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/3.67M [00:00<?, ?B/s]

In [29]:
tokenizer

GPT2Tokenizer(name_or_path='pop123ux/code-search-net-tokenizer', vocab_size=52000, model_max_length=1024, padding_side='right', truncation_side='right', special_tokens={'bos_token': '<|endoftext|>', 'eos_token': '<|endoftext|>', 'unk_token': '<|endoftext|>'}, added_tokens_decoder={
	0: AddedToken("<|endoftext|>", rstrip=False, lstrip=False, single_word=False, normalized=True, special=True),
})